In [ ]:
import pandas as pd
import re

In [ ]:
df = pd.read_csv(
    "Reviews.csv",
    usecols=['Summary', 'Text', 'Score'],
    on_bad_lines='skip',
    engine='python'
)

In [ ]:
df.shape

(568454, 3)

In [ ]:
df['review_text'] = df['Summary'].fillna('') + " " + df['Text'].fillna('')

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

In [ ]:
df['cleaned_review'] = df['review_text'].apply(clean_text)

In [ ]:
df

,Score,Summary,Text,review_text,cleaned_review
0,5,Good Quality Dog Food,I have bought several of the Vitality canned d...,Good Quality Dog Food I have bought several of...,good quality dog food i have bought several of...
1,1,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,Not as Advertised Product arrived labeled as J...,not as advertised product arrived labeled as j...
2,4,"""Delight"" says it all",This is a confection that has been around a fe...,"""Delight"" says it all This is a confection tha...",delight says it all this is a confection that ...
3,2,Cough Medicine,If you are looking for the secret ingredient i...,Cough Medicine If you are looking for the secr...,cough medicine if you are looking for the secr...
4,5,Great taffy,Great taffy at a great price. There was a wid...,Great taffy Great taffy at a great price. The...,great taffy great taffy at a great price ther...
...,...,...,...,...,...
568449,5,Will not do without,Great for sesame chicken..this is a good if no...,Will not do without Great for sesame chicken.....,will not do without great for sesame chickenth...
568450,2,disappointed,I'm disappointed with the flavor. The chocolat...,disappointed I'm disappointed with the flavor....,disappointed im disappointed with the flavor t...
568451,5,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o...",Perfect for our maltipoo These stars are small...,perfect for our maltipoo these stars are small...
568452,5,Favorite Training and reward treat,These are the BEST treats for training and rew...,Favorite Training and reward treat These are t...,favorite training and reward treat these are t...


In [ ]:
df.isnull().sum()

,0
Score,0
Summary,27
Text,0
review_text,0
cleaned_review,0


In [ ]:
df.dropna()

,Score,Summary,Text,review_text,cleaned_review
0,5,Good Quality Dog Food,I have bought several of the Vitality canned d...,Good Quality Dog Food I have bought several of...,good quality dog food i have bought several of...
1,1,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,Not as Advertised Product arrived labeled as J...,not as advertised product arrived labeled as j...
2,4,"""Delight"" says it all",This is a confection that has been around a fe...,"""Delight"" says it all This is a confection tha...",delight says it all this is a confection that ...
3,2,Cough Medicine,If you are looking for the secret ingredient i...,Cough Medicine If you are looking for the secr...,cough medicine if you are looking for the secr...
4,5,Great taffy,Great taffy at a great price. There was a wid...,Great taffy Great taffy at a great price. The...,great taffy great taffy at a great price ther...
...,...,...,...,...,...
568449,5,Will not do without,Great for sesame chicken..this is a good if no...,Will not do without Great for sesame chicken.....,will not do without great for sesame chickenth...
568450,2,disappointed,I'm disappointed with the flavor. The chocolat...,disappointed I'm disappointed with the flavor....,disappointed im disappointed with the flavor t...
568451,5,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o...",Perfect for our maltipoo These stars are small...,perfect for our maltipoo these stars are small...
568452,5,Favorite Training and reward treat,These are the BEST treats for training and rew...,Favorite Training and reward treat These are t...,favorite training and reward treat these are t...


In [ ]:
def sentiment_label(score):
    if score <= 2:
        return 0
    elif score == 3:
        return 1
    else:
        return 2

In [ ]:
df['sentiment'] = df['Score'].apply(sentiment_label)

In [ ]:
x = df['cleaned_review']        # input text (features)
y= df['sentiment']    # output labels


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42,stratify=df['sentiment'])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1, 2)
)

In [ ]:
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=1000
)

In [ ]:
lr.fit(x_train_tfidf,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=1000, multi_class='multinomial')

In [ ]:
y_pred_lr=lr.predict(x_test_tfidf)
y_pred_lr

array([2, 2, 2, ..., 2, 2, 2])

In [ ]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=["Negative", "Neutral", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

Accuracy: 0.8903519187974422

Classification Report:
              precision    recall  f1-score   support

    Negative       0.79      0.76      0.77     16407
     Neutral       0.60      0.28      0.38      8528
    Positive       0.92      0.97      0.95     88756

    accuracy                           0.89    113691
   macro avg       0.77      0.67      0.70    113691
weighted avg       0.88      0.89      0.88    113691


Confusion Matrix:
[[12420   761  3226]
 [ 1833  2399  4296]
 [ 1507   843 86406]]


In [ ]:
from sklearn.naive_bayes import MultinomialNB
nb=MultinomialNB()

In [ ]:
nb.fit(x_train_tfidf,y_train)

MultinomialNB()

In [ ]:
y_pred_nb=nb.predict(x_test_tfidf)
y_pred_nb

array([2, 2, 2, ..., 2, 2, 2])

In [ ]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_nb))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_nb,
    target_names=["Negative", "Neutral", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

Accuracy: 0.8451856347468137

Classification Report:
              precision    recall  f1-score   support

    Negative       0.82      0.47      0.60     16407
     Neutral       0.62      0.05      0.10      8528
    Positive       0.85      0.99      0.91     88756

    accuracy                           0.85    113691
   macro avg       0.76      0.51      0.54    113691
weighted avg       0.83      0.85      0.81    113691


Confusion Matrix:
[[ 7730   164  8513]
 [  923   464  7141]
 [  739   121 87896]]


In [ ]:
from sklearn.svm import LinearSVC
svm=LinearSVC()

In [ ]:
svm.fit(x_train_tfidf,y_train)

LinearSVC()

In [ ]:
y_pred_svm=svm.predict(x_test_tfidf)
y_pred_svm

array([2, 2, 2, ..., 2, 2, 2])

In [ ]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [1]:
print("Accuracy:", accuracy_score(y_test, y_pred_svm))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=["Negative", "Neutral", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

NameError: name 'model' is not defined